# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:**  Maria Alejandra Conde Pecho
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [1]:
# Archivos Landsat sobre el Peru (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat
def procesar_nombre_landsat(nombre_archivo):
    partes = nombre_archivo.replace(".TIF", "").split("_")

    sensor      = partes[0]
    path_row    = partes[2]
    fecha_raw   = partes[3]
    banda       = partes[7]

    path = path_row[:3]
    row  = path_row[3:]

    fecha = f"{fecha_raw[:4]}-{fecha_raw[4:6]}-{fecha_raw[6:]}"

    return {
        "sensor": sensor,
        "path": path,
        "row": row,
        "fecha": fecha,
        "banda": banda
    }


# Parte B: funcion clasificar_banda

def clasificar_banda(banda):
    tabla = {
        "B1": "Aerosol costero",
        "B5": "Infrarrojo cercano (NIR)"
    }
    return tabla.get(banda, "Desconocida")


# Parte C: procesar todos los archivos

fechas_unicas = set()

print("Resumen de escenas Landsat sobre Perú")

for archivo in archivos:
    info = procesar_nombre_landsat(archivo)
    nombre_banda = clasificar_banda(info["banda"])

    print(f"Fecha: {info['fecha']} | Path: {info['path']} | "
          f"Row: {info['row']} | Banda: {info['banda']} ({nombre_banda})")

    fechas_unicas.add(info["fecha"])

print(f"Total de fechas únicas: {len(fechas_unicas)}")

Resumen de escenas Landsat sobre Perú
Fecha: 2024-06-15 | Path: 008 | Row: 067 | Banda: B4 (Desconocida)
Fecha: 2024-06-15 | Path: 008 | Row: 067 | Banda: B5 (Infrarrojo cercano (NIR))
Fecha: 2024-07-01 | Path: 008 | Row: 067 | Banda: B3 (Desconocida)
Fecha: 2024-06-15 | Path: 009 | Row: 067 | Banda: B4 (Desconocida)
Fecha: 2024-07-08 | Path: 008 | Row: 067 | Banda: B6 (Desconocida)
Fecha: 2024-07-01 | Path: 008 | Row: 067 | Banda: B4 (Desconocida)
Total de fechas únicas: 3


---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [10]:
# Inventario forestal - Madre de Dios, Peru (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",      35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",            28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis",   -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",        18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",         52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",    8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",         45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",    62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",      41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",             22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",      5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",            38.9, 21.7],   # Bolaina
]

import math
# Parte A: Escribe la función calcular_area_basal aqui:
def calcular_area_basal(dap_cm):
    if dap_cm <= 0 or dap_cm == -999:
        return None

    ab = math.pi * (dap_cm / 200) ** 2
    return ab

# Parte B: Escribe la función clasificar_arbol aquí:
def clasificar_arbol(dap_cm, altura_m):
    if dap_cm < 10:
        clase = "Brinzal"
    elif dap_cm < 25:
        clase = "Latizal"
    elif dap_cm < 50:
        clase = "Fustal menor"
    else:
        clase = "Fustal mayor"

    # Alerta por valores sospechosos/erróneos
    alerta = (
        dap_cm > 200 or
        altura_m > 60 or
        (altura_m < 1 and dap_cm > 10)
    )
    return {"clase": clase, "alerta": alerta}

# Parte C: Procesar los datos e imprimir los resultados
arboles_validos = 0
area_basal_total = 0
conteo_clases = {"Brinzal": 0, "Latizal": 0, "Fustal menor": 0, "Fustal mayor": 0}
registros_alerta = 0

print("Procesamiento del inventario forestal - Madre de Dios (SERFOR)")

for registro in datos_arboles:
    id_arbol, especie, dap, altura = registro

    # Omitir árboles con datos faltantes (DAP o altura == -999)
    if dap == -999 or altura == -999:
        print(f"ID {id_arbol} ({especie}): OMITIDO (dato faltante -999)")
        continue

    ab = calcular_area_basal(dap)
    if ab is None:
        print(f"ID {id_arbol} ({especie}): OMITIDO (DAP inválido)")
        continue

    info = clasificar_arbol(dap, altura)

    # Acumular estadísticas
    arboles_validos += 1
    area_basal_total += ab
    conteo_clases[info["clase"]] += 1

    linea = (f"ID {id_arbol} ({especie}): DAP={dap} cm, altura={altura} m, "
              f"AB={ab:.5f} m², clase={info['clase']}")

    if info["alerta"]:
        registros_alerta += 1
        linea += "  ⚠️ ALERTA: valor atípico detectado"

    print(linea)

print("ESTADÍSTICAS DESCRIPTIVAS")
print(f"Número total de árboles válidos: {arboles_validos}")
print(f"Área basal total: {area_basal_total:.5f} m²")
print("Árboles por clase de tamaño:")
for clase, cantidad in conteo_clases.items():
    print(f"  - {clase}: {cantidad}")
print(f"Número de registros marcados con alerta: {registros_alerta}")

Procesamiento del inventario forestal - Madre de Dios (SERFOR)
ID 1 (Swietenia macrophylla): DAP=35.4 cm, altura=22.1 m, AB=0.09842 m², clase=Fustal menor
ID 2 (Cedrela odorata): DAP=28.2 cm, altura=18.5 m, AB=0.06246 m², clase=Fustal menor
ID 3 (Cedrelinga cateniformis): OMITIDO (dato faltante -999)
ID 4 (Virola surinamensis): DAP=18.7 cm, altura=12.3 m, AB=0.02746 m², clase=Latizal
ID 5 (Dipteryx micrantha): DAP=52.1 cm, altura=24.8 m, AB=0.21319 m², clase=Fustal mayor
ID 6 (Calycophyllum spruceanum): DAP=8.5 cm, altura=6.2 m, AB=0.00567 m², clase=Brinzal
ID 7 (Terminalia oblonga): DAP=45.0 cm, altura=85.0 m, AB=0.15904 m², clase=Fustal menor  ⚠️ ALERTA: valor atípico detectado
ID 8 (Cedrelinga cateniformis): DAP=62.3 cm, altura=28.4 m, AB=0.30484 m², clase=Fustal mayor
ID 9 (Swietenia macrophylla): OMITIDO (dato faltante -999)
ID 10 (Hura crepitans): DAP=22.5 cm, altura=0.5 m, AB=0.03976 m², clase=Latizal  ⚠️ ALERTA: valor atípico detectado
ID 11 (Schizolobium parahybum): DAP=5.2 cm

---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido